# 1、模型的profile属性的使用
举例：OpenRouter官网的模型

In [ ]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from rich import print as rprint
# 从.env文件中加载环境变量
load_dotenv(override=True)
model = ChatOpenRouter(model="openai/gpt-4o-mini",
# model="deepseek/deepseek-v3.2",
temperature=0.7,
timeout=30,
max_tokens=1000,
max_retries=6
)
rprint(model.profile)

# 2、模型初始化的完整参数
举例1：

In [ ]:
from langchain_deepseek import ChatDeepSeek
print(ChatDeepSeek.model_fields.keys())

举例2：使用init_chat_model

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv(override=True)
# 1. 实例化一个你感兴趣的模型对象
# 即使不传入具体 key，通常也能初始化成功
temp_model = init_chat_model(
model="deepseek-v4-flash",
model_provider="deepseek",
)
# 2. 现在它已经是一个具体的 ChatDeepSeek 对象了
# 你可以使用你熟悉的 .model_fields.keys()
print(temp_model.model_fields.keys())

举例3：测试model_kwargs参数的使用
langchain没有直接列出来的字段，但是模型本身支持，那么我们就可以把这些参数声明在model_kwargs参数中

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    model_kwargs={
        "tools": [
            {
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "description": "Get weather of a location, the user should supply a location first.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. SanFrancisco, CA",
                            }
                        },
                        "required": ["location"],
                    },
                },
            }
        ]
    },
)

# 向模型发送单条数据
response = model.invoke("你好，今天北京的天气如何")
# 打印响应
rprint(response)

举例4：extra_body  用于存放模型厂商基于OpenAI API协议扩展的字段

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(
model="deepseek:deepseek-v4-flash",
extra_body={"thinking": {"type": "disabled"}},
)
# 向模型发送单条数据
response = model.invoke("你好，一句话回答")
# 打印响应
rprint(response)

# 3、模型调用（invoke、stream、batch等）中config参数
举例

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 1. 初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    temperature=0.2,
    max_tokens=500,
    # 指定可调整参数
    configurable_fields=(
        "model",
        "model_provider",
        "temperature",
        "max_tokens",
    ),
)

# 2. 准备 config 字典
config = {
    "run_name": "joke_generation",  # 在LangSmith中这次运行会显示为 joke_generation
    "tags": ["tag1", "tag2"],  # 打上标签便于分类查找
    "metadata": {"user_id": "123"},  # 记录用户ID
    "configurable": {
        "model": "deepseek-v4-pro",  # 配置模型参数
        "model_provider": "openai",  # 配置模型提供商参数
        "temperature": 0.7,  # 配置温度参数
        "max_tokens": 1000,  # 配置最大令牌数
    },
}

# 3. 调用模型并传入config
response = model.invoke(
    "1 + 2 = ？",
    config=config
)

rprint(response)